In [2]:
import pandas as pd
from google.colab import drive

# 1. Montar o drive (necessário rodar antes de ler os arquivos)
drive.mount('/content/drive')

# 2. Carregar os dados (note o /content/drive/ antes do MyDrive)
df_clusters = pd.read_csv('/content/drive/MyDrive/projeto_ciencia_de_dados/dados_projeto_ciencia_dados/painel_uf_clusters.csv')
df_integrado = pd.read_csv('/content/drive/MyDrive/projeto_ciencia_de_dados/dados_projeto_ciencia_dados/painel_uf_integrado.csv')
df_mensal = pd.read_csv('/content/drive/MyDrive/projeto_ciencia_de_dados/dados_projeto_ciencia_dados/painel_mensal_uf.csv')

Mounted at /content/drive


In [ ]:
# 3. Mesclar clusters + integrado (relação 1:1). Remove colunas duplicadas (sufixos _x/_y).
cols_to_drop_integrado = [c for c in df_integrado.columns if c in df_clusters.columns and c != 'uf']
df_integrado_clean = df_integrado.drop(columns=cols_to_drop_integrado)
df_merged = pd.merge(df_clusters, df_integrado_clean, on='uf', how='left')

# 4. IMPORTANTE: estas colunas existem tanto no nível anual (integrado) quanto no mensal.
#    Para a página de Sazonalidade precisamos da versão MENSAL (que varia por mês),
#    então removemos as versões anuais antes de mesclar o mensal — caso contrário o
#    consumo fica constante nos 12 meses e as barras saem todas iguais.
cols_mensais = ['consumo_comercial_mwh', 'consumo_industrial_mwh',
                'consumo_residencial_mwh', 'n_domicilios', 'consumo_resid_por_domicilio']
df_merged = df_merged.drop(columns=[c for c in cols_mensais if c in df_merged.columns])

# 5. Mesclar com o mensal (relação 1:N) — agora as versões mensais prevalecem.
cols_to_drop_mensal = [c for c in df_mensal.columns if c in df_merged.columns and c != 'uf']
df_mensal_clean = df_mensal.drop(columns=cols_to_drop_mensal)
df_final = pd.merge(df_merged, df_mensal_clean, on='uf', how='left')

# 6. Salvar o dataframe consolidado de volta no seu Drive
output_filename = '/content/drive/MyDrive/projeto_ciencia_de_dados/dados_projeto_ciencia_dados/dados_consolidados_looker.csv'
df_final.to_csv(output_filename, index=False)

print(f"Arquivo consolidado gerado com sucesso em: {output_filename}")
print(f"Colunas: {list(df_final.columns)}")
print(f"Tamanho: {df_final.shape}")
